# **Processamento de Linguagem Natural [2026-Q3]**
Prof. Alexandre Donizeti Alves

### **PROJETO PRÁTICO** [LangChain + Grandes Modelos de Linguagem]


O **PROJETO PRÁTICO** deve ser feito utilizando o **Google Colab** com uma conta sua vinculada ao Gmail. O link do seu notebook armazenado no Google Drive e o link de um repositório no GitHub devem ser enviados usando o seguinte formulário:

> https://forms.gle/D4gLqP1iGgyn2hbH8


**IMPORTANTE**: A submissão deve ser feita até o dia **04/12 (sexta-feira)** APENAS POR UM INTEGRANTE DA EQUIPE, até às 23h59. Por favor, lembre-se de dar permissão de ACESSO IRRESTRITO para o professor da disciplina.

### **EQUIPE**

---

**Integrante 01:**

Guilherme Araujo dos Santos — RA: 11202020294

**Integrante 02:**

Ian Victor Toniolo Silva — RA: 11202020351

### **GRANDE MODELO DE LINGUAGEM (*Large Language Model - LLM*)**

---

Cada equipe deve selecionar um Grande Modelo de Linguagem (*Large Language Model - LMM*).



Por favor, informe os dados do LLM selecionado:

**LLM**: Claude Haiku 5.5 (Anthropic), identificador na API: `claude-haiku-5-5`

**Link para a documentação oficial**: https://docs.claude.com/en/docs/about-claude/models/overview

### **API (Opcional)**
---

Por favor, informe os dados da API selecionada:

**API**: MediaWiki Action API (Wikipédia em português)

**Site oficial**: https://www.mediawiki.org

**Link para a documentação oficial**: https://www.mediawiki.org/wiki/API:Main_page

### **DESCRIÇÃO**
---

Implementar um `notebook` no `Google Colab` que faça uso do framework **`LangChain`** (obrigatório) e de um **LLM** aplicando, no mínimo, DUAS técnicas de PLN. As técnicas podem ser aplicada em qualquer córpus obtido a partir de uma **API** ou a partir de uma página Web.

O **LLM** e a **API** selecionados devem ser informados na seguinte planilha:

> https://docs.google.com/spreadsheets/d/1iIUZcwnywO7RuF6VEJ8Rx9NDT1cwteyvsnkhYr0NWtU/edit?usp=sharing

>
As seguintes técnicas de PLN podem ser usadas:

*   Correção Gramatical
*   Classificação de Textos
*   Análise de Sentimentos
*   Detecção de Emoções
*   Extração de Palavras-chave
*   Tradução de Textos
*   Sumarização de Textos
*   Similaridade de Textos
*   Reconhecimento de Entidades Nomeadas
*   Sistemas de Perguntas e Respostas
>

**IMPORTANTE:** É obrigatório usar o e-mail da UFABC.


### **CRITÉRIOS DE AVALIAÇÃO**
---


Serão considerados como critérios de avaliação os seguintes pontos:

* Uso do framework **`LangChain`**.

* Escolha e uso de um **LLM**.

* Escolha e uso de uma **API** ou **Página Web**.

* Projeto disponível no Github.

* Apresentação (10 a 15 minutos).

* Criatividade no uso do framework **`LangChain`** em conjunto com o **LLM** e a **API**.



Também é interessante explorar técnicas como:
*   Geração de Dados Sintéticos
*   Geração Aumentada por Recuperação (RAG)
*   Agentes


**IMPORTANTE**: todo o código do notebook deve ser documentado e EXECUTADO. Código sem execução não será considerado.

### **IMPLEMENTAÇÃO**
---

### Visão geral
O usuário informa um **tema**. O notebook:
1. Busca o artigo mais relevante na **Wikipédia** via **MediaWiki Action API** (`list=search` + `prop=extracts`);
2. Usa **LangChain** + **Claude Haiku 5.5** para aplicar três técnicas de PLN sobre o texto do artigo:
   - **Sumarização de Textos**
   - **Reconhecimento de Entidades Nomeadas (NER)**, com saída estruturada (Pydantic)
   - **Sistema de Perguntas e Respostas** baseado no artigo (o modelo responde só com o que está no texto recuperado — uma forma simples de RAG)

### 1. Instalação das dependências

In [1]:
# langchain-anthropic: integração do LangChain com os modelos Claude
# requests: chamadas HTTP para a MediaWiki API
# python-dotenv: lê a chave de um arquivo .env quando o notebook roda fora do Colab
%pip install -q langchain langchain-anthropic requests pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


### 2. Configuração da chave da API da Anthropic
A chave **não** fica no código.
- **No Colab:** clique no ícone de chave (🔑 *Secrets*) na barra lateral, crie um segredo chamado `ANTHROPIC_API_KEY` com a sua chave do Claude Console e habilite o acesso do notebook.
- **Localmente (VS Code/Jupyter):** crie um arquivo `.env` na pasta do projeto com a linha `ANTHROPIC_API_KEY=...`. Ele está no `.gitignore` e não vai para o repositório.

In [2]:
import os

try:
    from google.colab import userdata  # rodando no Colab
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    from dotenv import load_dotenv  # rodando localmente
    load_dotenv()

print("Chave carregada:", bool(os.environ.get("ANTHROPIC_API_KEY")))

Chave carregada: True


### 3. Coleta do córpus: MediaWiki Action API
Duas chamadas ao endpoint `https://pt.wikipedia.org/w/api.php`:
- `action=query&list=search` → encontra o título do artigo mais relevante para o tema;
- `action=query&prop=extracts&explaintext=1` → baixa o conteúdo do artigo em texto puro.

A Wikimedia exige um cabeçalho `User-Agent` identificando o cliente.

In [3]:
import requests

WIKI_API = "https://pt.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "PLN-UFABC-ProjetoPratico/1.0 (projeto academico)"}


def consultar_api(params: dict) -> dict:
    """Chama a MediaWiki API e devolve o JSON; erros da API viram exceção com a mensagem original."""
    resp = requests.get(WIKI_API, params={**params, "format": "json"}, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    dados = resp.json()
    if "error" in dados:
        erro = dados["error"]
        raise RuntimeError(f"MediaWiki API: {erro.get('code')}: {erro.get('info')}")
    if "query" not in dados:
        raise RuntimeError(f"Resposta inesperada da MediaWiki API: {str(dados)[:500]}")
    return dados["query"]


def buscar_titulo(tema: str) -> str:
    """Retorna o título do artigo da Wikipédia mais relevante para o tema."""
    resultados = consultar_api({
        "action": "query",
        "list": "search",
        "srsearch": tema,
        "srlimit": 1,
    })["search"]
    if not resultados:
        raise ValueError(f"Nenhum artigo encontrado para: {tema}")
    return resultados[0]["title"]


def obter_texto(titulo: str, max_chars: int = 15000) -> str:
    """Baixa o texto puro do artigo (limitado a max_chars para controlar custo/tokens)."""
    paginas = consultar_api({
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,
        "titles": titulo,
        "redirects": 1,
    })["pages"]
    texto = next(iter(paginas.values())).get("extract", "")
    return texto[:max_chars]


# Teste rápido
titulo = buscar_titulo("Universidade Federal do ABC")
artigo = obter_texto(titulo)
print("Artigo:", titulo)
print("Tamanho:", len(artigo), "caracteres\n")
print(artigo[:800])

Artigo: Universidade Federal do ABC
Tamanho: 15000 caracteres

A Universidade Federal do ABC (UFABC) é uma instituição pública federal de ensino superior no ABC Paulista. Ocupa o 1° lugar entre as universidades brasileiras no Ranking SCImago nos quesitos “Excelência em Pesquisa”, “Publicações de alta qualidade” e “Impacto normalizado das suas publicações”. Foi avaliada pelo Índice Geral de Cursos (IGC) do MEC como a melhor universidade do Estado de São Paulo, sendo avaliada como a 1ª no ranking de cursos de graduação entre todas as universidades do Brasil. O IGC leva em consideração em sua avaliação fatores como infraestrutura, corpo docente, e nota dos formandos no ENADE. Ocupa o 1º lugar entre as universidades brasileiras no quesito "Internacionalização" no Ranking Universitário Folha 2019.
O presidente da comissão que formulou a proposta da univers


### 4. LLM via LangChain
`ChatAnthropic` é o wrapper do LangChain para os modelos Claude. O Claude Haiku 5.5 não aceita o parâmetro `temperature` (a API devolve erro 400), então usamos os valores padrão de amostragem. `max_tokens=4096` dá folga para o raciocínio interno do modelo, que conta dentro desse limite, e para a saída estruturada do NER.

In [4]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatAnthropic(model="claude-haiku-5-5", max_tokens=4096)

# Teste de conexão
print(llm.invoke("Responda em uma frase: o que é PLN?").content)

PLN (Processamento de Linguagem Natural) é a área da inteligência artificial e da linguística computacional que permite que computadores compreendam, interpretem e gerem textos e falas humanas.


### 5. Técnica 1: Sumarização de Textos
Uma *chain* LCEL (`prompt | llm | parser`): o prompt recebe o artigo e pede um resumo em português, em tópicos.

In [5]:
prompt_resumo = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente que resume artigos enciclopédicos em português do Brasil, de forma fiel ao texto."),
    ("human", "Resuma o artigo abaixo em até 5 tópicos curtos.\n\nTítulo: {titulo}\n\nArtigo:\n{artigo}"),
])

chain_resumo = prompt_resumo | llm | StrOutputParser()

resumo = chain_resumo.invoke({"titulo": titulo, "artigo": artigo})
print(resumo)

- **Identificação:** a UFABC é uma instituição pública federal de ensino superior do ABC Paulista, criada por lei de 2005, com campi em Santo André (sede) e São Bernardo do Campo.

- **Destaques:** segundo o texto, a universidade lidera o Ranking SCImago em pesquisa e publicações, foi considerada a melhor universidade paulista pelo IGC do MEC e ficou em 1º lugar em "Internacionalização" no Ranking Universitário Folha 2019. É a única federal com 100% de professores doutores.

- **Ensino:** oferece bacharelados interdisciplinares (Ciência e Tecnologia; Ciências e Humanidades) organizados em quadrimestres, o que permite cursar mais disciplinas por ano. Os egressos podem seguir para 21 cursos de graduação ou para o mercado de trabalho. Em 2024 foram aprovados três novos cursos de licenciatura, e o curso de Ciência de Dados começa em 2025.

- **Pesquisa e pós-graduação:** em 2007, a CAPES aprovou 10 dos 11 pedidos de cursos de pós-graduação stricto sensu. Em 2009, a universidade foi a terce

### 6. Técnica 2: Reconhecimento de Entidades Nomeadas (NER)
Usamos `with_structured_output` com um esquema **Pydantic**: o LangChain força o modelo a devolver uma lista de entidades já tipadas (PESSOA, ORGANIZAÇÃO, LOCAL, DATA, OUTRO), pronta para virar tabela.

O prompt define cada tipo com exemplos (inclusive datas). Como o modelo nem sempre respeita uma quantidade pedida no prompt, a função `extrair_entidades` limita o resultado a 25 entidades no código, escolhendo em rodízio entre os tipos para que todos os tipos encontrados no texto apareçam na tabela.

In [6]:
from typing import List, Literal
from pydantic import BaseModel, Field
import pandas as pd


class Entidade(BaseModel):
    texto: str = Field(description="A entidade exatamente como aparece no texto")
    tipo: Literal["PESSOA", "ORGANIZACAO", "LOCAL", "DATA", "OUTRO"]


class ListaEntidades(BaseModel):
    entidades: List[Entidade] = Field(description="Entidades nomeadas distintas, ordenadas da mais relevante para a menos relevante")


prompt_ner = ChatPromptTemplate.from_messages([
    ("system",
     "Você é um sistema de Reconhecimento de Entidades Nomeadas. Classifique cada entidade em um tipo:\n"
     "- PESSOA: nomes de pessoas\n"
     "- ORGANIZACAO: instituições, empresas, órgãos públicos, veículos de imprensa\n"
     "- LOCAL: cidades, estados, países, regiões, endereços\n"
     "- DATA: datas e anos citados no texto (ex.: \"2005\", \"26 de julho de 2005\")\n"
     "- OUTRO: demais nomes próprios (leis, prêmios, cursos, obras, produtos)\n"
     "Não repita entidades. Ordene a lista da entidade mais relevante para a menos relevante, "
     "e garanta que as primeiras posições cubram todos os tipos que aparecem no texto."),
    ("human", "Extraia as entidades nomeadas do texto abaixo.\n\n{artigo}"),
])

chain_ner = prompt_ner | llm.with_structured_output(ListaEntidades)


def extrair_entidades(artigo: str, max_entidades: int = 25) -> list[Entidade]:
    """Roda a chain de NER e devolve até `max_entidades` entidades, equilibradas entre os tipos.

    O limite é aplicado no código porque o modelo nem sempre respeita uma quantidade pedida no prompt.
    A seleção é feita em rodízio entre os tipos (mantendo a ordem de relevância do modelo dentro de
    cada tipo), para que nenhum tipo encontrado no texto fique de fora."""
    todas = chain_ner.invoke({"artigo": artigo}).entidades
    por_tipo = {}
    for e in todas:
        por_tipo.setdefault(e.tipo, []).append(e)
    selecionadas = []
    while len(selecionadas) < max_entidades and any(por_tipo.values()):
        for fila in por_tipo.values():
            if fila and len(selecionadas) < max_entidades:
                selecionadas.append(fila.pop(0))
    return selecionadas


entidades = extrair_entidades(artigo)
df_entidades = pd.DataFrame([e.model_dump() for e in entidades])
df_entidades.sort_values("tipo").reset_index(drop=True)

,texto,tipo
0,2007,DATA
1,27 de julho de 2005,DATA
2,2004,DATA
3,26 de julho de 2005,DATA
4,2005,DATA
5,Santo André,LOCAL
6,Grande ABC,LOCAL
7,São Bernardo do Campo,LOCAL
8,ABC Paulista,LOCAL
9,Estado de São Paulo,LOCAL


In [7]:
# Contagem de entidades por tipo
df_entidades["tipo"].value_counts()

tipo
ORGANIZACAO    5
PESSOA         5
OUTRO          5
LOCAL          5
DATA           5
Name: count, dtype: int64

### 7. Técnica 3: Sistema de Perguntas e Respostas
O artigo recuperado da API é injetado no prompt como **contexto**, e o modelo é instruído a responder **apenas** com base nele (Geração Aumentada por Recuperação simples). Se a resposta não estiver no artigo, ele deve dizer isso em vez de inventar.

In [8]:
prompt_qa = ChatPromptTemplate.from_messages([
    ("system",
     "Responda à pergunta usando SOMENTE o contexto fornecido (um artigo da Wikipédia). "
     "Se a resposta não estiver no contexto, responda: 'O artigo não traz essa informação.' "
     "Responda em português, de forma direta."),
    ("human", "Contexto:\n{artigo}\n\nPergunta: {pergunta}"),
])

chain_qa = prompt_qa | llm | StrOutputParser()

perguntas = [
    "Em que ano a universidade foi criada?",
    "Em quais cidades ficam os campi?",
    "Quem ganhou a Copa do Mundo de 2002?",  # fora do contexto: deve recusar
]

for p in perguntas:
    print("P:", p)
    print("R:", chain_qa.invoke({"artigo": artigo, "pergunta": p}), "\n")

P: Em que ano a universidade foi criada?


R: A UFABC foi criada em **2005**, pela Lei nº 11.145, datada de 26 de julho de 2005 e publicada no Diário Oficial da União em 27 de julho de 2005. 

P: Em quais cidades ficam os campi?


R: Os campi da UFABC estão sediados nos municípios de Santo André (sede) e São Bernardo do Campo. 

P: Quem ganhou a Copa do Mundo de 2002?


R: O artigo não traz essa informação. 



### 8. Pipeline completo: qualquer tema
Junta tudo em uma função: tema → Wikipédia → resumo + entidades + respostas.

In [9]:
def analisar_tema(tema: str, perguntas: list[str]):
    titulo = buscar_titulo(tema)
    artigo = obter_texto(titulo)
    print(f"=== Artigo: {titulo} ({len(artigo)} caracteres) ===\n")

    print("--- RESUMO ---")
    print(chain_resumo.invoke({"titulo": titulo, "artigo": artigo}), "\n")

    print("--- ENTIDADES NOMEADAS ---")
    ents = extrair_entidades(artigo)
    display(pd.DataFrame([e.model_dump() for e in ents]))

    print("\n--- PERGUNTAS E RESPOSTAS ---")
    for p in perguntas:
        print("P:", p)
        print("R:", chain_qa.invoke({"artigo": artigo, "pergunta": p}), "\n")


analisar_tema("Alan Turing", [
    "Onde Alan Turing nasceu?",
    "Qual foi a contribuição dele durante a Segunda Guerra Mundial?",
])

=== Artigo: Alan Turing (15000 caracteres) ===

--- RESUMO ---


- **Quem foi:** matemático, cientista da computação, lógico, criptoanalista e filósofo britânico, considerado o pai da ciência da computação teórica e da inteligência artificial.
- **Contribuição teórica:** em 1936, com o artigo "Sobre números computáveis", formalizou os conceitos de algoritmo e computação por meio da máquina de Turing, modelo de computador de uso geral.
- **Segunda Guerra Mundial:** trabalhou em Bletchley Park na quebra das cifras alemãs, incluindo a Enigma, e liderou a Hut 8, responsável pela criptoanálise naval. Estima-se que seu trabalho tenha encurtado a guerra na Europa em mais de dois anos e salvado mais de 14 milhões de vidas.
- **Pós-guerra:** projetou o Mecanismo de Computação Automática no Laboratório Nacional de Física, ajudou a desenvolver os computadores de Manchester e fez estudos sobre biologia matemática, incluindo a morfogênese.
- **Perseguição e legado:** foi processado em 1952 por atos homossexuais e aceitou castração química como alternativa à pris

,texto,tipo
0,Alan Mathison Turing,PESSOA
1,Londres,LOCAL
2,23 de junho de 1912,DATA
3,Lei de Segredos Oficiais,OUTRO
4,Escola de Código e Cifras do Governo (GC&CS),ORGANIZACAO
5,Max Newman,PESSOA
6,Wilmslow,LOCAL
7,7 de junho de 1954,DATA
8,Segunda Guerra Mundial,OUTRO
9,Hut 8,ORGANIZACAO



--- PERGUNTAS E RESPOSTAS ---
P: Onde Alan Turing nasceu?


R: Alan Turing nasceu em Maida Vale, Londres, em 23 de junho de 1912. 

P: Qual foi a contribuição dele durante a Segunda Guerra Mundial?


R: Durante a Segunda Guerra Mundial, Turing trabalhou em Bletchley Park, centro britânico de criptoanálise, onde liderou por um tempo a Hut 8, responsável pela análise criptográfica naval alemã. Ele desenvolveu técnicas para acelerar a quebra das cifras alemãs, incluindo melhorias no método polonês de bombardeio e uma máquina eletromecânica capaz de encontrar configurações para a máquina Enigma. Também teve papel crucial na decodificação de mensagens interceptadas, o que ajudou os Aliados a vencer batalhas decisivas, como a Batalha do Atlântico. Estima-se que esse trabalho tenha encurtado a guerra na Europa em mais de dois anos e salvo mais de 14 milhões de vidas. 



### 9. Conclusão
- **API:** a MediaWiki Action API forneceu o córpus (artigos da Wikipédia em português) sob demanda, a partir de qualquer tema.
- **LangChain:** usado para montar as *chains* (prompt → LLM → parser) e para obter saída estruturada com Pydantic.
- **LLM:** o Claude Haiku 5.5 aplicou três técnicas de PLN: sumarização, NER e perguntas e respostas fundamentadas no texto recuperado.